In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from mamba_ssm import Mamba

/home/imvikranth/miniconda3/envs/mamba310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
from collections import defaultdict

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [4]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [5]:
class VideoFrameDataset(Dataset):

    def __init__(self, root_dir, transform=None, frames_per_video=7):

        self.samples = []
        self.transform = transform

        classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {c:i for i,c in enumerate(classes)}

        for cls in classes:

            cls_path = os.path.join(root_dir, cls)
            videos = defaultdict(list)

            for img in os.listdir(cls_path):
                video_name = img.split("_f")[0]
                videos[video_name].append(img)

            for v in videos:

                frames = sorted(videos[v])[:frames_per_video]

                if len(frames) < frames_per_video:
                    continue

                paths = [os.path.join(cls_path,f) for f in frames]

                self.samples.append((paths, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        paths, label = self.samples[idx]

        frames = []

        for p in paths:

            img = Image.open(p).convert("RGB")

            if self.transform:
                img = self.transform(img)

            frames.append(img)

        frames = torch.stack(frames)

        return frames, label

In [6]:
train_dataset = VideoFrameDataset(
    root_dir="/home/imvikranth/mini_project/frame_dataset/train",
    transform=transform,
)

val_dataset = VideoFrameDataset(
    root_dir="/home/imvikranth/mini_project/frame_dataset/val",
    transform=transform,
)

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False ,
)

num_classes = len(train_dataset.class_to_idx)
print("Number of classes:", num_classes)
print(num_classes , type(num_classes))

Number of classes: 101
101 <class 'int'>


In [9]:
class MambaImageClassifier(nn.Module):

    def __init__(self, num_classes, d_model=128, patch_size=16):
        super().__init__()

        self.patch_embed = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)

        self.mamba = Mamba(
            d_model=d_model,
            d_state=16,
            d_conv=4,
            expand=2,
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(int(d_model),int(num_classes))

    def forward(self, x):   

        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.mamba(x)
        x = self.norm(x)
        x = x.mean(dim=1)

        return self.head(x)

In [10]:
frame_model = MambaImageClassifier(num_classes)
frame_model.load_state_dict(torch.load("mamba_image_model.pth"))
frame_model = frame_model.to(device)
frame_model.eval()

MambaImageClassifier(
  (patch_embed): Conv2d(3, 128, kernel_size=(16, 16), stride=(16, 16))
  (mamba): Mamba(
    (in_proj): Linear(in_features=128, out_features=512, bias=False)
    (conv1d): Conv1d(256, 256, kernel_size=(4,), stride=(1,), padding=(3,), groups=256)
    (act): SiLU()
    (x_proj): Linear(in_features=256, out_features=40, bias=False)
    (dt_proj): Linear(in_features=8, out_features=256, bias=True)
    (out_proj): Linear(in_features=256, out_features=128, bias=False)
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=128, out_features=101, bias=True)
)

In [11]:
class MambaImageEncoder(nn.Module):

    def __init__(self, frame_model):
        super().__init__()

        self.patch_embed = frame_model.patch_embed
        self.mamba = frame_model.mamba
        self.norm = frame_model.norm

    def forward(self, x):

        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1,2)

        x= x.contiguous()  

        x = self.mamba(x)
        x = self.norm(x)
        x = x.mean(dim=1)

        return x

In [12]:
encoder = MambaImageEncoder(frame_model).to(device)

for param in encoder.parameters():
    param.requires_grad = False 

In [16]:
class TemporalMambaClassifier(nn.Module):

    def __init__(self, num_classes, d_model=128):
        super().__init__()

        self.encoder = encoder 

        self.temporal_mamba = Mamba(
            d_model=d_model,
            d_state=16,
            d_conv=4,
            expand=2
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):

        B,T,C,H,W = x.shape

        x = x.view(B*T, C, H, W)
        x = x.to(device)

        features = self.encoder(x)
        features = features.view(B, T, -1)
        features = features.contiguous()

        temporal_features = self.temporal_mamba(features)
        temporal_features = self.norm(temporal_features)

        video_feature = temporal_features.mean(dim=1)

        return self.head(video_feature)

In [17]:
model = TemporalMambaClassifier(num_classes).to(device)

criterion = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model.parameters(), lr=3e-4)

In [18]:
epochs = 20 

for epoch in range(epochs):
    model.train()
    total_loss = 0 

    for videos,labels in train_loader :
        videos = videos.to(device) 
        labels = labels.to(device) 

        optimizer.zero_grad() 
        outputs = model(videos) 

        loss = criterion(outputs,labels) 

        loss.backward() 
        optimizer.step()

        total_loss+= loss.item()
    print(f"Epoch{epoch+1},Loss:{total_loss/len(train_loader):.4f}")
    

Epoch1,Loss:2.5083
Epoch2,Loss:0.6885
Epoch3,Loss:0.2756
Epoch4,Loss:0.1332
Epoch5,Loss:0.0922
Epoch6,Loss:0.0565
Epoch7,Loss:0.0512
Epoch8,Loss:0.0526
Epoch9,Loss:0.0371
Epoch10,Loss:0.0383
Epoch11,Loss:0.0438
Epoch12,Loss:0.0277
Epoch13,Loss:0.0320
Epoch14,Loss:0.0346
Epoch15,Loss:0.0279
Epoch16,Loss:0.0223
Epoch17,Loss:0.0401
Epoch18,Loss:0.0266
Epoch19,Loss:0.0219
Epoch20,Loss:0.0313


Epoch1,Loss:2.5083
Epoch2,Loss:0.6885
Epoch3,Loss:0.2756
Epoch4,Loss:0.1332
Epoch5,Loss:0.0922
Epoch6,Loss:0.0565
Epoch7,Loss:0.0512
Epoch8,Loss:0.0526
Epoch9,Loss:0.0371
Epoch10,Loss:0.0383
Epoch11,Loss:0.0438
Epoch12,Loss:0.0277
Epoch13,Loss:0.0320
Epoch14,Loss:0.0346
Epoch15,Loss:0.0279
Epoch16,Loss:0.0223
Epoch17,Loss:0.0401
Epoch18,Loss:0.0266
Epoch19,Loss:0.0219
Epoch20,Loss:0.0313

In [19]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for videos, labels in val_loader:

        videos = videos.to(device)
        labels = labels.to(device)

        outputs = model(videos)

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.8386132695756127


it tool 31m 2s and accuracy 0.8386132695756127


In [20]:
torch.save(model.state_dict(), "temporal_mamba.pth")